Part 2: Simple Logistic Regression Model

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from pathlib import Path

splits_dir = Path("Data/splits")

# Indlæser filer
train = pd.read_csv(splits_dir / "train.csv")
val = pd.read_csv(splits_dir / "val.csv")
test = pd.read_csv(splits_dir / "test.csv")

# Mapping af labels:
labels = {
    'reliable': 'reliable', 'political': 'reliable',
    'fake': 'fake', 'unreliable': 'fake', 'conspiracy': 'fake',
    'bias': 'fake', 'hate': 'fake', 'clickbait': 'fake'
}

for df in [train, val, test]:
    df["binary_label"] = df["type"].map(labels)

train = train.dropna(subset=['binary_label', "processed_content"])
val = val.dropna(subset=['binary_label', "processed_content"])
test = test.dropna(subset=['binary_label', "processed_content"])


# Sets up model candidates, however, since we have strict requirements in part 2 only one is used
candidates = [
    {
        "name": "CountVectorizer_10k_C1",
        "vectorizer": CountVectorizer(max_features=10000),
        "model": LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
    },
    # {
    #     "name": "CountVectorizer_20k_C1",
    #     "vectorizer": CountVectorizer(max_features=20000),
    #     "model": LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0) 
    # }
]

best_name = None
best_vectorizer = None
best_model = None
best_val_f1 = -1


# Finds best model from val data
for candidate in candidates:
    vectorizer = candidate["vectorizer"]
    model = candidate["model"]

    X_train = vectorizer.fit_transform(train["processed_content"])
    X_val = vectorizer.transform(val["processed_content"])

    model.fit(X_train, train["binary_label"])
    val_preds = model.predict(X_val)

    val_f1 = f1_score(val["binary_label"], val_preds, pos_label="fake")

    print(f"\nValidation results for {candidate['name']}:")
    print(classification_report(val["binary_label"], val_preds, zero_division=0))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_name = candidate["name"]
        best_vectorizer = vectorizer
        best_model = model

print(f"\nBest model chosen on validation set: {best_name}")
print(f"Best validation F1 (fake): {best_val_f1:.4f}")


# Runs the best model on the test data
X_test = best_vectorizer.transform(test['processed_content'])
test_preds = best_model.predict(X_test)

print("Final results on test set:")
print(classification_report(test['binary_label'], test_preds, zero_division=0))

c:\Users\malth\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Validation results for CountVectorizer_10k_C1:
              precision    recall  f1-score   support

        fake       0.84      0.89      0.86     40500
    reliable       0.88      0.84      0.86     41308

    accuracy                           0.86     81808
   macro avg       0.86      0.86      0.86     81808
weighted avg       0.86      0.86      0.86     81808


Best model chosen on validation set: CountVectorizer_10k_C1
Best validation F1 (fake): 0.8639
Final results on test set:
              precision    recall  f1-score   support

        fake       0.84      0.89      0.86     40498
    reliable       0.88      0.84      0.86     41309

    accuracy                           0.86     81807
   macro avg       0.86      0.86      0.86     81807
weighted avg       0.86      0.86      0.86     81807



In [6]:
from Data_Processing import preprocess_text

liar_mappings = {
    "true": "reliable",
    "mostly-true": "reliable",
    "half-true": "reliable",
    "barely-true": "fake",
    "false": "fake",
    "pants-fire": "fake"
}

test_liar = pd.read_csv(splits_dir / "LIAR_test.tsv", sep='\t', header=None)
test_liar["binary_label"] = test_liar[1].map(liar_mappings)

test_liar["processed_content"] = test_liar[2].fillna("").apply(preprocess_text)

test_liar = test_liar.dropna(subset=["binary_label", "processed_content"])

X_LIAR = best_vectorizer.transform(test_liar["processed_content"])
liar_preds = best_model.predict(X_LIAR)

print("\n LIAR test results:")
print(classification_report(test_liar["binary_label"], liar_preds, zero_division=0))


 LIAR test results:
              precision    recall  f1-score   support

        fake       0.44      0.89      0.59       553
    reliable       0.57      0.11      0.19       714

    accuracy                           0.45      1267
   macro avg       0.50      0.50      0.39      1267
weighted avg       0.51      0.45      0.36      1267

